In [ ]:
import pandas as pd
import numpy as np
from chronological_split import chronological_split 
import statsmodels.api as sm


df = pd.read_csv("../data/preprocessed.csv")

train_set , test_set =  chronological_split(df,train_size = 0.8)




train_set['HOME/AWAY']= np.where(train_set['MATCHUP'].str.contains('@'), 'Away', 'Home')
test_set['HOME/AWAY']=  np.where(test_set['MATCHUP'].str.contains('@'), 'Away', 'Home')

train_set['HOME/AWAY'] = pd.Categorical(train_set['OPPONENT'])
test_set['HOME/AWAY'] = pd.Categorical(test_set['OPPONENT'])


train_set['OPPONENT'] = train_set['MATCHUP'].str[-3:]
train_set['OPPONENT'] = pd.Categorical(train_set['OPPONENT'])


                                     
test_set['OPPONENT'] = test_set['MATCHUP'].str[-3:]
test_set['OPPONENT'] = pd.Categorical(test_set['OPPONENT'])

X_train = train_set[['PREV_GAME_PTS', 'AVG_PTS_LAG5', 'REST_DAYS' , 'OPPONENT' , 'HOME/AWAY' ]]
X_test = test_set[['PREV_GAME_PTS', 'AVG_PTS_LAG5', 'REST_DAYS' , 'OPPONENT' , 'HOME/AWAY' ]]

y_train , y_test = train_set['PTS'] , test_set['PTS']






In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first",sparse_output= False), ["OPPONENT", 'HOME/AWAY']),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

Linear Regression (OLS)

In [58]:
X_train_ols = sm.add_constant(X_train_processed)
X_test_ols = sm.add_constant(X_test_processed)

ols = sm.OLS(y_train, X_train_ols).fit()

y_pred = ols.predict(X_test_ols)

rmse = root_mean_squared_error(y_test, y_pred) 
mae = mean_absolute_error(y_test,y_pred)
r_2 = r2_score(y_test,y_pred)

print('Performance Metrics:')
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R^2: {r_2:.2f}")

ols.params

Performance Metrics:
RMSE: 5.16
MAE: 3.91
R^2: 0.67


const                       1.013406
cat__OPPONENT_BKN          -0.499805
cat__OPPONENT_BOS          -0.606221
cat__OPPONENT_CHA          -0.467065
cat__OPPONENT_CHI          -0.431297
                              ...   
cat__HOME/AWAY_UTA         -0.190195
cat__HOME/AWAY_WAS         -0.298358
remainder__PREV_GAME_PTS   -0.219074
remainder__AVG_PTS_LAG5     1.217965
remainder__REST_DAYS       -0.015708
Length: 62, dtype: float64

Random Forest

In [60]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_processed,y_train)

in_sample_pred =  rf.predict(X_train_processed)


train_rmse = root_mean_squared_error(y_train, in_sample_pred) 
train_mae = mean_absolute_error(y_train,in_sample_pred)
train_r_2 = r2_score(y_train,in_sample_pred)

print('Performance Metrics:')
print(f"RMSE: {train_rmse:.2f}")
print(f"MAE: {train_mae:.2f}")
print(f"R^2: {train_r_2:.2F}")


y_pred = rf.predict(X_test_processed)


rmse = root_mean_squared_error(y_test, y_pred) 
mae = mean_absolute_error(y_test,y_pred)
r_2 = r2_score(y_test,y_pred)

print('Performance Metrics:')
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R^2: {r_2:.2F}")


importances = rf.feature_importances_

feature_names = preprocessor.get_feature_names_out()

pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

Performance Metrics:
RMSE: 4.80
MAE: 3.64
R^2: 0.72
Performance Metrics:
RMSE: 5.21
MAE: 3.94
R^2: 0.66


,Feature,Importance
59,remainder__AVG_PTS_LAG5,0.963166
58,remainder__PREV_GAME_PTS,0.026618
60,remainder__REST_DAYS,0.003821
10,cat__OPPONENT_IND,0.000209
39,cat__HOME/AWAY_IND,0.000200
...,...,...
3,cat__OPPONENT_CHI,0.000066
20,cat__OPPONENT_ORL,0.000065
22,cat__OPPONENT_PHX,0.000064
32,cat__HOME/AWAY_CHI,0.000053


Negative Binomial Regression

In [57]:
#There is overdispersion...
print(f"Mean for 'PTS': {np.round(df["PTS"].mean(),2)}")
print(f"Variance for 'PTS': {np.round(df["PTS"].var(),2)}")

Mean for 'PTS': 10.91
Variance for 'PTS': 81.32


In [61]:
import warnings
from statsmodels.tools.sm_exceptions import HessianInversionWarning

warnings.filterwarnings(
    "ignore",
    category=HessianInversionWarning
)

X_train_nb = sm.add_constant(X_train_processed)
X_test_nb = sm.add_constant(X_test_processed)

init_model = sm.NegativeBinomial(y_train, X_train_nb)
init_results = init_model.fit(disp=0) # disp=0 suppresses output
optimal_alpha = init_results.params['alpha']

print(f"Estimated Alpha (Dispersion): {optimal_alpha}")

glm_nb_model = sm.GLM(y_train, X_train_nb, 
                      family=sm.families.NegativeBinomial(alpha=optimal_alpha))
glm_results = glm_nb_model.fit()

y_pred = glm_results.predict(X_test_nb)


rmse = root_mean_squared_error(y_test, y_pred) 
mae = mean_absolute_error(y_test,y_pred)
r_2 = r2_score(y_test,y_pred)

print('Performance Metrics:')
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R^2: {r_2:.2f}")
print(glm_results.params)



Estimated Alpha (Dispersion): 0.2481183340370502
Performance Metrics:
RMSE: 7.29
MAE: 4.84
R^2: 0.34
const                       1.342196
cat__OPPONENT_BKN          -0.065038
cat__OPPONENT_BOS          -0.070377
cat__OPPONENT_CHA          -0.053445
cat__OPPONENT_CHI          -0.047169
                              ...   
cat__HOME/AWAY_UTA         -0.041249
cat__HOME/AWAY_WAS         -0.054879
remainder__PREV_GAME_PTS   -0.018966
remainder__AVG_PTS_LAG5     0.108743
remainder__REST_DAYS       -0.010876
Length: 62, dtype: float64


In [51]:
print(glm_results.params)


const                       1.342196
cat__OPPONENT_BKN          -0.065038
cat__OPPONENT_BOS          -0.070377
cat__OPPONENT_CHA          -0.053445
cat__OPPONENT_CHI          -0.047169
                              ...   
cat__HOME/AWAY_UTA         -0.041249
cat__HOME/AWAY_WAS         -0.054879
remainder__PREV_GAME_PTS   -0.018966
remainder__AVG_PTS_LAG5     0.108743
remainder__REST_DAYS       -0.010876
Length: 62, dtype: float64
